In [1]:
import os
import time
import cv2
import torch
import numpy as np
import pandas as pd
from difflib import SequenceMatcher
from tqdm import tqdm
from ultralytics import YOLO
import torchvision.transforms as transforms
from transformers import Swin2SRImageProcessor, Swin2SRForImageSuperResolution


# 1. LOAD MODELS

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# YOLO
yolo = YOLO("runs/detect/train7/weights/best.pt")

# Swin2SR
base_model_name = "caidas/swin2SR-realworld-sr-x4-64-bsrgan-psnr"
sr_processor = Swin2SRImageProcessor.from_pretrained(base_model_name)
sr_model = Swin2SRForImageSuperResolution.from_pretrained(base_model_name).to(device)
sr_model.load_state_dict(torch.load("swin2sr_v2_plate_epoch2.pth", map_location=device))
sr_model.eval()

# Custom OCR CNN
class OCR_CNN_Deep(torch.nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.features = torch.nn.Sequential(
            torch.nn.Conv2d(1, 32, 3, padding=1), torch.nn.BatchNorm2d(32), torch.nn.ReLU(),
            torch.nn.Conv2d(32, 32, 3, padding=1), torch.nn.BatchNorm2d(32), torch.nn.ReLU(),
            torch.nn.MaxPool2d(2), torch.nn.Dropout(0.25),

            torch.nn.Conv2d(32, 64, 3, padding=1), torch.nn.BatchNorm2d(64), torch.nn.ReLU(),
            torch.nn.Conv2d(64, 64, 3, padding=1), torch.nn.BatchNorm2d(64), torch.nn.ReLU(),
            torch.nn.MaxPool2d(2), torch.nn.Dropout(0.25),

            torch.nn.Conv2d(64, 128, 3, padding=1), torch.nn.BatchNorm2d(128), torch.nn.ReLU(),
            torch.nn.Conv2d(128, 128, 3, padding=1), torch.nn.BatchNorm2d(128), torch.nn.ReLU(),
            torch.nn.MaxPool2d(2), torch.nn.Dropout(0.25),

            torch.nn.Conv2d(128, 256, 3, padding=1), torch.nn.BatchNorm2d(256), torch.nn.ReLU(),
            torch.nn.Conv2d(256, 256, 3, padding=1), torch.nn.BatchNorm2d(256), torch.nn.ReLU(),
            torch.nn.MaxPool2d(2), torch.nn.Dropout(0.25),
        )
        self.classifier = torch.nn.Sequential(
            torch.nn.Flatten(),
            torch.nn.Linear(256 * 2 * 2, 512), torch.nn.ReLU(), torch.nn.Dropout(0.5),
            torch.nn.Linear(512, num_classes)
        )

    def forward(self, x):
        return self.classifier(self.features(x))

ocr_model = OCR_CNN_Deep(num_classes=36).to(device)
ocr_model.load_state_dict(torch.load("ocr_model_deep_v1.pth", map_location=device))
ocr_model.eval()

class_names = list("0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZ")
AMBIGUOUS = {"0": "O", "O": "0", "1": "I", "I": "1", "8": "B", "B": "8"}

transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Grayscale(),
    transforms.Resize((32, 32)),
    transforms.ToTensor()
])


# 2. UTILITY FUNCTIONS

def is_red_background(char_rgb):
    hsv = cv2.cvtColor(char_rgb, cv2.COLOR_RGB2HSV)
    mask1 = cv2.inRange(hsv, (0, 70, 50), (10, 255, 255))
    mask2 = cv2.inRange(hsv, (170, 70, 50), (180, 255, 255))
    red_mask = cv2.bitwise_or(mask1, mask2)
    return (red_mask.mean() / 255.0) > 0.15

def preprocess_dark_on_dark(char_rgb):
    gray = cv2.cvtColor(char_rgb, cv2.COLOR_RGB2GRAY)
    return cv2.normalize(gray, None, 0, 255, cv2.NORM_MINMAX)

def is_dark_on_dark(char_rgb):
    gray = cv2.cvtColor(char_rgb, cv2.COLOR_RGB2GRAY)
    return gray.mean() < 80 and (gray.max() - gray.min()) < 60

def upscale_with_swin(img_rgb):
    h_orig, w_orig = img_rgb.shape[:2]
    inputs = sr_processor(img_rgb, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = sr_model(**inputs)

    output_img = outputs.reconstruction.data.cpu().float().clamp(0, 1).numpy()[0]
    output_img = np.moveaxis(output_img, 0, -1)
    
    # Crop garbage padding (enforcing x4 logic)
    output_img = output_img[:h_orig * 4, :w_orig * 4, :]
    return (output_img * 255.0).round().astype(np.uint8)

def predict_character(char_img_rgb):
    if is_red_background(char_img_rgb):
        char_img_rgb = cv2.bitwise_not(cv2.cvtColor(char_img_rgb, cv2.COLOR_RGB2GRAY))
    elif is_dark_on_dark(char_img_rgb):
        char_img_rgb = preprocess_dark_on_dark(char_img_rgb)
    
    char_img = transform(char_img_rgb).unsqueeze(0).to(device)

    with torch.no_grad():
        out = ocr_model(char_img)
        conf, pred = torch.softmax(out, dim=1).max(dim=1)

    return class_names[pred.item()], conf.item()

def disambiguate_plate(chars):
    chars = chars.copy()
    for i, ch in enumerate(chars):
        if ch not in AMBIGUOUS: continue
        if i < 3 and ch.isdigit(): chars[i] = AMBIGUOUS[ch]
        elif i >= 3 and ch.isalpha(): chars[i] = AMBIGUOUS[ch]
    return chars

def sort_boxes_reading_order(boxes, confs):
    chars = [{"box": (x1, y1, x2, y2), "conf": c, "cx": (x1+x2)/2, "cy": (y1+y2)/2, "h": y2-y1} 
             for (x1, y1, x2, y2), c in zip(boxes, confs)]
    avg_h = np.mean([c["h"] for c in chars])
    rows = []
    
    for c in chars:
        placed = False
        for row in rows:
            if abs(c["cy"] - row[0]["cy"]) < avg_h * 0.6:
                row.append(c)
                placed = True
                break
        if not placed: rows.append([c])

    rows.sort(key=lambda r: np.mean([c["cy"] for c in r]))
    for row in rows: row.sort(key=lambda c: c["cx"])

    return [c["box"] for row in rows for c in row], [c["conf"] for row in rows for c in row]

def filter_false_positives(boxes, confs, img_h):
    if len(boxes) == 0: return [], []
    
    heights, valid_indices = [], []
    for i, (_, y1, _, y2) in enumerate(boxes):
        h = y2 - y1
        if h >= img_h * 0.10: # Remove tiny specks
            heights.append(h)
            valid_indices.append(i)

    if not valid_indices: return [], []

    median_h = np.median(heights)
    final_boxes, final_confs = [], []
    for idx, h in zip(valid_indices, heights):
        if 0.5 * median_h < h < 1.5 * median_h:
            final_boxes.append(boxes[idx])
            final_confs.append(confs[idx])

    return final_boxes, final_confs


# 3. MAIN PIPELINE

def recognize_plate_with_yolo_ocr(image_path, conf=0.35):
    img_bgr = cv2.imread(image_path)
    if img_bgr is None: return ""
    
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    h_orig, w_orig = img_rgb.shape[:2]

    results = yolo(img_bgr, conf=conf, iou=0.3, verbose=False)
    if len(results[0].boxes) == 0: return ""

    boxes, confs = results[0].boxes.xyxy.cpu().numpy(), results[0].boxes.conf.cpu().numpy()
    boxes, confs = filter_false_positives(boxes, confs, h_orig)
    if len(boxes) == 0: return ""

    boxes, confs = sort_boxes_reading_order(boxes, confs)
    chars = []

    for (x1, y1, x2, y2), _ in zip(boxes, confs):
        # 20% margin for Swin2SR context
        margin_x, margin_y = 0.20 * (x2 - x1), 0.20 * (y2 - y1)
        x1_m, y1_m = int(max(0, x1 - margin_x)), int(max(0, y1 - margin_y))
        x2_m, y2_m = int(min(w_orig, x2 + margin_x)), int(min(h_orig, y2 + margin_y))

        crop_orig = img_rgb[y1_m:y2_m, x1_m:x2_m]
        if crop_orig.size == 0: continue

        try:
            crop_sr = upscale_with_swin(crop_orig)
        except Exception:
            crop_sr = crop_orig

        # 10% trim to tighten crop for OCR
        h_sr, w_sr = crop_sr.shape[:2]
        trim_x, trim_y = int(0.10 * w_sr), int(0.10 * h_sr)
        crop_sr_tight = crop_sr[trim_y:h_sr - trim_y, trim_x:w_sr - trim_x]
        if crop_sr_tight.size == 0: crop_sr_tight = crop_sr

        ch, _ = predict_character(crop_sr_tight)
        chars.append(ch)

    return "".join(disambiguate_plate(chars))


# 4. EVALUATION

def char_accuracy(pred, gt):
    return sum(p == g for p, g in zip(pred, gt)), len(gt)

def cer(pred, gt):
    if len(gt) == 0: return 1.0
    return 1 - SequenceMatcher(None, pred, gt).ratio()

def evaluate_ocr_batch(recognize_fn, labels_df, image_dir, conf=0.35):
    total_plates, fully_correct = 0, 0
    total_gt_chars, total_correct_chars = 0, 0
    no_detection, length_mismatch, partial_match = 0, 0, 0
    cer_scores, times = [], []

    for _, row in tqdm(labels_df.iterrows(), total=len(labels_df), desc="Evaluating"):
        img_path = os.path.join(image_dir, row["filename"])
        gt_plate = row["plate"].strip()

        start = time.perf_counter()
        pred_plate = recognize_fn(img_path, conf)
        times.append(time.perf_counter() - start)
        
        total_plates += 1

        if pred_plate == "":
            no_detection += 1
            cer_scores.append(1.0)
            total_gt_chars += len(gt_plate)
            continue

        if pred_plate == gt_plate:
            fully_correct += 1
        else:
            if len(pred_plate) != len(gt_plate): length_mismatch += 1
            else: partial_match += 1

        correct, gt_len = char_accuracy(pred_plate, gt_plate)
        total_correct_chars += correct
        total_gt_chars += gt_len
        cer_scores.append(cer(pred_plate, gt_plate))

    # Calculate metrics
    plate_accuracy = fully_correct / total_plates if total_plates else 0
    char_accuracy_score = total_correct_chars / total_gt_chars if total_gt_chars else 0
    mean_cer = np.mean(cer_scores) if cer_scores else 0

    print("\n===== OCR EVALUATION RESULTS =====")
    print(f"Total plates           : {total_plates}")
    print(f"Fully correct plates   : {fully_correct}")
    print(f"Plate accuracy         : {plate_accuracy:.4f}")
    print("\n--- Character-level ---")
    print(f"Total GT characters    : {total_gt_chars}")
    print(f"Correct characters     : {total_correct_chars}")
    print(f"Character accuracy     : {char_accuracy_score:.4f}")
    print(f"Mean CER               : {mean_cer:.4f}")
    print("\n--- Error breakdown ---")
    print(f"No detection           : {no_detection}")
    print(f"Length mismatch        : {length_mismatch}")
    print(f"Partial match          : {partial_match}")
    print("\n--- Timing ---")
    print(f"Avg time / plate (s)   : {np.mean(times):.4f}")
    print(f"Std time / plate (s)   : {np.std(times):.4f}")

    return {
        "plate_accuracy": plate_accuracy,
        "char_accuracy": char_accuracy_score,
        "mean_cer": mean_cer,
        "avg_time": np.mean(times),
        "std_time": np.std(times)
    }


# EXECUTION

if __name__ == "__main__":
    labels_df = pd.read_csv( " ") # path to labels
    image_dir = " " # path to UFPR-ALPR Datset or a custom one

    results_plate_sr = evaluate_ocr_batch(
        recognize_fn=recognize_plate_with_yolo_ocr,
        labels_df=labels_df,
        image_dir=image_dir,
        conf=0.35
    )

Evaluating: 100%|██████████████████████████████████████████████████████████████████| 1800/1800 [34:43<00:00,  1.16s/it]


===== OCR EVALUATION RESULTS =====
Total plates           : 1800
Fully correct plates   : 737
Plate accuracy         : 0.4094

--- Character-level ---
Total GT characters    : 12600
Correct characters     : 9196
Character accuracy     : 0.7298
Mean CER               : 0.2584

--- Error breakdown ---
No detection           : 0
Length mismatch        : 13
Partial match          : 1050

--- Timing ---
Avg time / plate (s)   : 1.1560
Std time / plate (s)   : 0.3350
